###Config

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

###Read the label map + prepare model

In [ ]:
# Colab: run this in a Python cell
import sys, subprocess, pkgutil

# Uninstall all OpenCV variants + numpy
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y",
                "opencv-python", "opencv-contrib-python",
                "opencv-python-headless", "numpy"])


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'uninstall', '-y', 'opencv-python', 'opencv-contrib-python', 'opencv-python-headless', 'numpy'], returncode=0)

In [ ]:
# Reinstall fresh, pin NumPy to a version OpenCV supports (<2.3)
!python -m pip install --no-cache-dir "numpy>=2,<2.3"

# Pick ONE OpenCV flavor (headless is best for Colab)
!python -m pip install "opencv-python-headless<5"

# Make spaCy/Thinc happy with NumPy 2
!python -m pip install -U thinc spacy

# Optional: sanity check resolver
!python -m pip check


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 69.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
albucore 0.0.24 requires opencv-python-headless>=4.9.0.80, which is not installed.
albumentations 2.0.8 requires opencv-python-headless>=4.9.0.80, which is not installed.
dopamine-rl 4.1.2 requires opencv-python>=3.4.8.29, which is not installed.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.2.6 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.2.6 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 MB 11.7 MB/s eta 0:00:00
dopamine-rl 4.1.2 requires opencv-python, which is not installed.
ipython 7.34.0 requires jedi, which is not installed.
numba 0.60.0 has requirement numpy

In [ ]:
!pip -q install transformers==4.42.4 accelerate==0.33.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 85.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.1/315.1 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 82.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires opencv-python>=3.4.8.29, which is not installed.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.


In [ ]:
import os, json, math, gc
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel


In [ ]:
BASE        = "/content/drive/MyDrive/msc_dissertation"
HYBRID_CSV  = f"{BASE}/summaries_hybrid.csv"
LABEL_MAP   = f"{BASE}/clause_to_idx.json"            # maps clause -> idx
CKPT_PATH   = f"{BASE}/legalpro_bert_clause_cls_best.pt"  # your trained weights
OUT_DIR     = f"{BASE}/pipeline_out"
os.makedirs(OUT_DIR, exist_ok=True)

MODEL_NAME      = "AmitTewari/LegalPro-BERT-base"
MAX_LEN         = 512
BATCH_SIZE      = 8
CHUNK_OVERLAP   = 64
AGGREGATION     = "max"
THRESHOLD_MODE  = "global"
GLOBAL_THR      = 0.50
TOP_K_EXPORT    = 10
DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE


device(type='cuda')

####Load label map, tokenizer, and model

In [ ]:
with open(LABEL_MAP, "r") as f:
    clause_to_idx = json.load(f)
idx_to_clause = {v: k for k, v in clause_to_idx.items()}
NUM_LABELS    = len(clause_to_idx)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
class LegalProBertClauseClassifier(nn.Module):
    def __init__(self, model_name=MODEL_NAME, num_labels=NUM_LABELS, dropout=0.3):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        self.dropout  = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.backbone.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        logits = self.classifier(self.dropout(cls))
        return logits

In [ ]:
!python -m pip uninstall -y transformers tokenizers huggingface-hub safetensors accelerate


Found existing installation: transformers 4.42.4
Uninstalling transformers-4.42.4:
  Successfully uninstalled transformers-4.42.4
Found existing installation: tokenizers 0.19.1
Uninstalling tokenizers-0.19.1:
  Successfully uninstalled tokenizers-0.19.1
Found existing installation: huggingface-hub 0.34.4
Uninstalling huggingface-hub-0.34.4:
  Successfully uninstalled huggingface-hub-0.34.4
Found existing installation: safetensors 0.6.2
Uninstalling safetensors-0.6.2:
  Successfully uninstalled safetensors-0.6.2
Found existing installation: accelerate 0.33.0
Uninstalling accelerate-0.33.0:
  Successfully uninstalled accelerate-0.33.0


In [ ]:
!python -m pip install --no-cache-dir \
  "transformers==4.44.2" \
  "tokenizers>=0.19,<0.20" \
  "huggingface-hub>=0.24,<0.26" \
  "accelerate>=0.33" \
  "safetensors>=0.4.4"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 246.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 192.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 436.6/436.6 kB 338.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.8/485.8 kB 401.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.35.1 requires huggingface-hub>=0.34.0, but you have huggingface-hub 0.25.2 which is incompatible.
gradio 5.43.1 requires huggingface-hub<1.0,>=0.33.5, but you have huggingface-hub 0.25.2 which is incompatible.


In [ ]:
import transformers, tokenizers, huggingface_hub, accelerate, safetensors, torch
print("transformers:", transformers.__version__)
print("tokenizers:", tokenizers.__version__)
print("hub:", huggingface_hub.__version__)
print("accelerate:", accelerate.__version__)
print("safetensors:", safetensors.__version__)
print("torch:", torch.__version__)


transformers: 4.44.2
tokenizers: 0.19.1
hub: 0.25.2
accelerate: 1.10.1
safetensors: 0.6.2
torch: 2.8.0+cu126


In [ ]:
# load model + weights
model = LegalProBertClauseClassifier(num_labels=NUM_LABELS)
ckpt = torch.load(CKPT_PATH, map_location="cpu")
model.load_state_dict(ckpt if isinstance(ckpt, dict) else ckpt.state_dict(), strict=False)
model.to(DEVICE).eval()

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

LegalProBertClauseClassifier(
  (backbone): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps

####4) Data: read CSV and prepare text (use hybrid_blend)

In [ ]:
df = pd.read_csv(HYBRID_CSV)
assert "hybrid_blend" in df.columns, "Column 'hybrid_blend' not found"
texts = df["hybrid_blend"].fillna("").astype(str).tolist()
len(texts), df.columns.tolist()[:6]


(365, ['context', 'extractive_summary', 'pegasus_summary', 'hybrid_blend'])

####Chunking utilities (token-aware)

In [ ]:
def tokenize_to_ids(text):
    # no special truncation here; we need full ids to make chunks
    enc = tokenizer(text, add_special_tokens=False, return_attention_mask=False, return_tensors=None)
    return enc["input_ids"]

def chunk_ids(ids, max_len=MAX_LEN, overlap=CHUNK_OVERLAP):
    assert max_len > 2, "max_len must allow [CLS] and [SEP]"
    stride = max_len - overlap
    chunks = []
    for start in range(0, len(ids), stride):
        piece = ids[start:start + max_len - 2]  # reserve for [CLS],[SEP]
        if not piece:
            break
        chunks.append([tokenizer.cls_token_id] + piece + [tokenizer.sep_token_id])
    return chunks

def collate_batch(batch):
    # `batch` is a list of chunks, each chunk is a list of token ids
    # Do NOT flatten — pass a list of lists to pad
    enc = tokenizer.pad({"input_ids": batch}, padding=True, return_tensors="pt")
    return enc["input_ids"], enc["attention_mask"]



In [ ]:
def chunk_encode(text, max_len=MAX_LEN, overlap=CHUNK_OVERLAP):
    enc = tokenizer(
        text,
        max_length=max_len,
        truncation=True,
        stride=overlap,
        return_overflowing_tokens=True,
        return_attention_mask=True,
        padding=False,
        add_special_tokens=True,   # let tokenizer add [CLS]/[SEP]
    )
    # enc["input_ids"] is already a list of chunk sequences
    input_ids_list = enc["input_ids"]
    attn_list = enc["attention_mask"]

    if len(input_ids_list) == 0:
        input_ids_list = [[tokenizer.cls_token_id, tokenizer.sep_token_id]]
        attn_list = [[1, 1]]
    return input_ids_list, attn_list

####Batched inference over (variable) chunks per document

In [ ]:
@torch.inference_mode()
def predict_probs_for_texts_fast(texts):
    probs_per_doc = []
    for t in tqdm(texts, desc="Scoring hybrid_blend"):
        ids_list, attn_list = chunk_encode(t)

        # mini-batch over chunks
        chunk_probs = []
        for i in range(0, len(ids_list), BATCH_SIZE):
            batch_ids = ids_list[i:i+BATCH_SIZE]
            batch_att = attn_list[i:i+BATCH_SIZE]
            enc = tokenizer.pad({"input_ids": batch_ids, "attention_mask": batch_att},
                                padding=True, return_tensors="pt")
            logits = model(
                input_ids=enc["input_ids"].to(DEVICE),
                attention_mask=enc["attention_mask"].to(DEVICE)
            )
            chunk_probs.append(torch.sigmoid(logits).cpu().numpy())

        chunk_probs = np.vstack(chunk_probs)
        agg = chunk_probs.max(axis=0) if AGGREGATION == "max" else chunk_probs.mean(axis=0)
        probs_per_doc.append(agg)

    return np.vstack(probs_per_doc)

# Use this instead of the older function:
probs = predict_probs_for_texts_fast(texts)

Scoring hybrid_blend:   0%|          | 0/365 [00:00<?, ?it/s]

You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


####Thresholding -> binary predictions

In [ ]:
if THRESHOLD_MODE == "per_class":
    # (Option A) learn per-class thresholds from training (if you saved them).
    # Here we fallback to a simple heuristic: You can replace with your saved vector.
    per_class_thr = np.full((NUM_LABELS,), 0.5, dtype=np.float32)
    thrs = per_class_thr
else:
    # (Option B) global threshold
    thrs = np.full((NUM_LABELS,), GLOBAL_THR, dtype=np.float32)

y_pred = (probs >= thrs).astype(int)  # [N, L]
y_pred.shape


(365, 41)

In [ ]:
def topk_names(row_probs, k=TOP_K_EXPORT):
    idxs = np.argsort(row_probs)[::-1][:k]
    return "; ".join([f"{idx_to_clause[i]}:{row_probs[i]:.3f}" for i in idxs])

pred_df = pd.DataFrame({
    "doc_id": np.arange(len(df)),
    "topk_predicted_clauses": [topk_names(p, TOP_K_EXPORT) for p in probs],
    "probs_json": [json.dumps({idx_to_clause[i]: float(p[i]) for i in range(NUM_LABELS)}) for p in probs],
})

# (optional) attach original columns you care about
pred_df["hybrid_blend"] = df["hybrid_blend"].astype(str)

# also include wide binary columns per clause for easy analysis
for i in range(NUM_LABELS):
    pred_df[f"bin_{idx_to_clause[i]}"] = y_pred[:, i]

out_path = os.path.join(OUT_DIR, "clauses_legalprobert.csv")
pred_df.to_csv(out_path, index=False)
out_path


'/content/drive/MyDrive/msc_dissertation/pipeline_out/clauses_legalprobert.csv'

In [ ]:
pred_df.head(3)[["doc_id","topk_predicted_clauses"]]


,doc_id,topk_predicted_clauses
0,0,Ip Ownership Assignment:0.759; Liquidated Dama...
1,1,Liquidated Damages:0.712; Change Of Control:0....
2,2,Expiration Date:0.712; Liquidated Damages:0.68...


In [ ]:
# df  -> read from HYBRID_CSV earlier (has context + summaries)
# pred_df -> built in Step 8 (has doc_id, topk_predicted_clauses, probs_json, bin_*)

# ensure a doc_id column on the summaries side that matches pred_df
df_with_id = df.reset_index().rename(columns={"index": "doc_id"})

# columns from summaries you want to keep
summary_cols = [c for c in ["context", "extractive_summary", "pegasus_summary", "hybrid_blend"] if c in df_with_id.columns]

# drop duplicate 'hybrid_blend' from pred_df if you added it earlier
pred_slim = pred_df.drop(columns=[c for c in ["hybrid_blend"] if c in pred_df.columns])

combined = df_with_id[["doc_id"] + summary_cols].merge(pred_slim, on="doc_id", how="left")

# save CSV (+ optional Parquet for faster load later)
combined_path_csv = os.path.join(OUT_DIR, "summaries_with_clause_predictions.csv")
combined.to_csv(combined_path_csv, index=False)

combined_path_parquet = os.path.join(OUT_DIR, "summaries_with_clause_predictions.parquet")
combined.to_parquet(combined_path_parquet, index=False)

combined_path_csv


'/content/drive/MyDrive/msc_dissertation/pipeline_out/summaries_with_clause_predictions.csv'

In [ ]:
display_cols = ["doc_id", "hybrid_blend", "topk_predicted_clauses"]
combined[display_cols].head(3)


,doc_id,hybrid_blend,topk_predicted_clauses
0,0,The agreement and order 2.1 During the validit...,Ip Ownership Assignment:0.759; Liquidated Dama...
1,1,"As used in this Agreement, the term ""Web site""...",Liquidated Damages:0.712; Change Of Control:0....
2,2,Exhibit 1 The undersigned hereby agree to join...,Expiration Date:0.712; Liquidated Damages:0.68...


####Evaluation

In [ ]:
CANDIDATE_SUMMARY_COLS = [c for c in ["extractive_summary", "pegasus_summary", "hybrid_blend"] if c in df.columns]
assert "context" in df.columns, "df must have a 'context' column"
assert len(CANDIDATE_SUMMARY_COLS) > 0, "No summary columns found"

In [ ]:
try:
    clause_to_idx
except NameError:
    # you likely loaded this earlier from LABEL_MAP
    raise RuntimeError("Please load clause_to_idx from LABEL_MAP before running.")

In [ ]:
# critical, domain-driven clauses
try:
    CRITICAL_CLAUSES
except NameError:
    CRITICAL_CLAUSES = [
        "Governing Law",
        "Termination For Convenience",
        "Confidentiality",
        "Cap On Liability",
        "Ip Ownership Assignment",
        "Indemnification",
        "Arbitration",
        "Change of Control"
    ]

In [ ]:
def batched_probs(texts, max_len=512, bs=8):
    outs = []
    for i in range(0, len(texts), bs):
        toks = tokenizer(
            texts[i:i+bs], padding=True, truncation=True,
            max_length=max_len, return_tensors="pt"
        ).to(DEVICE)
        with torch.no_grad():
            logits = model(toks["input_ids"], toks["attention_mask"])
            outs.append(torch.sigmoid(logits).cpu().numpy())
    return np.vstack(outs)

In [ ]:
ctx_probs = batched_probs(df["context"].astype(str).tolist())

In [ ]:
def binarize(probs, thr):
    return (probs >= float(thr)).astype(int)

In [ ]:
def micro_macro(Y_true, Y_pred):
    from sklearn.metrics import classification_report
    rep = classification_report(Y_true, Y_pred, output_dict=True, zero_division=0)
    rep = pd.DataFrame(rep).transpose()
    return {
        "micro_p": rep.loc["micro avg","precision"],
        "micro_r": rep.loc["micro avg","recall"],
        "micro_f1":rep.loc["micro avg","f1-score"],
        "macro_f1":rep.loc["macro avg","f1-score"]
    }

In [ ]:
def ccs(y_full, y_sum):
    sel = (y_full==1)
    return np.nan if sel.sum()==0 else float((y_sum[sel]==1).sum()/sel.sum())

In [ ]:
def rcr(y_doc, y_sum, clause_to_idx, crit):
    idxs = [clause_to_idx[c] for c in crit if c in clause_to_idx]
    if not idxs: return np.nan
    want = (y_doc[idxs]==1)
    return np.nan if want.sum()==0 else float((y_sum[idxs]==1).sum()/want.sum())


In [ ]:
from sklearn.metrics import f1_score, precision_recall_fscore_support

In [ ]:
from sklearn.metrics import f1_score
grid = np.linspace(0.05, 0.95, 37)
best_gf1, best_gthr = -1.0, 0.30
for t in grid:
    y_pred = (ctx_probs >= t).astype(int)
    f1 = f1_score(y_pred.ravel(), y_pred.ravel(), average="micro", zero_division=0)  # self-comparison
    if f1 > best_gf1:
        best_gf1, best_gthr = f1, float(t)

# per-class thresholds (you can just use global threshold if you want simplicity)
C = ctx_probs.shape[1]
best_thr = np.full(C, best_gthr, dtype=float)

# pseudo-gold labels
Y_ctx = (ctx_probs >= best_thr).astype(int)

In [ ]:
def evaluate_summary_against_context(col_name):
    texts = df[col_name].astype(str).tolist()
    sum_probs = batched_probs(texts)
    Y_sum = (sum_probs >= best_thr).astype(int)

    # Clause Coverage Score (agreement with context positives)
    CCS = [ccs(Y_ctx[i], Y_sum[i]) for i in range(len(df))]
    RCR_critical = [rcr(Y_ctx[i], Y_sum[i], clause_to_idx, CRITICAL_CLAUSES) for i in range(len(df))]
    compression = (
        df[col_name].astype(str).str.split().str.len() /
        df["context"].astype(str).str.split().str.len().clip(lower=1)
    )

    # Summary stats
    results = {
        "CCS_mean": float(np.nanmean(CCS)),
        "RCR_critical_mean": float(np.nanmean(RCR_critical)),
        "compression_mean": float(np.nanmean(compression)),
        "n_docs": len(df)
    }
    return results

reports = {}
for col in ["extractive_summary", "pegasus_summary", "hybrid_blend"]:
    if col in df.columns:
        reports[col] = evaluate_summary_against_context(col)

pd.DataFrame(reports).to_csv(f"{OUT_DIR}/pseudo_gold_eval.csv")
reports


{'extractive_summary': {'CCS_mean': 1.0,
  'RCR_critical_mean': 1.0,
  'compression_mean': 0.3226465739738618,
  'n_docs': 365},
 'pegasus_summary': {'CCS_mean': 1.0,
  'RCR_critical_mean': 1.0,
  'compression_mean': 0.05238089562862376,
  'n_docs': 365},
 'hybrid_blend': {'CCS_mean': 1.0,
  'RCR_critical_mean': 1.0,
  'compression_mean': 0.12009479048266761,
  'n_docs': 365}}

In [ ]:
!pip install rouge-score==0.1.2 bert-score==0.3.13


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.3 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=ba97d64b982a360a72014709a35f95e5a32e2848efaffa0cea2bbfda2e738e90
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [ ]:
from rouge_score import rouge_scorer
from bert_score import score as bertscore


In [ ]:
def compute_rouge(cands, refs):
    scorer = rouge_scorer.RougeScorer(["rouge1","rouge2","rougeLsum"], use_stemmer=True)
    agg = {k:[] for k in ["r1_p","r1_r","r1_f","r2_p","r2_r","r2_f","rl_p","rl_r","rl_f"]}
    for c,r in zip(cands, refs):
        if not c or not r:
            for k in agg: agg[k].append(0.0)
            continue
        s = scorer.score(r, c)
        agg["r1_p"].append(s["rouge1"].precision); agg["r1_r"].append(s["rouge1"].recall); agg["r1_f"].append(s["rouge1"].fmeasure)
        agg["r2_p"].append(s["rouge2"].precision); agg["r2_r"].append(s["rouge2"].recall); agg["r2_f"].append(s["rouge2"].fmeasure)
        agg["rl_p"].append(s["rougeLsum"].precision); agg["rl_r"].append(s["rougeLsum"].recall); agg["rl_f"].append(s["rougeLsum"].fmeasure)
    return {k: float(np.mean(v)) for k,v in agg.items()}


In [ ]:
# --- choose reference texts ---
if "reference_summary" in df.columns:
    refs = df["reference_summary"].fillna("").astype(str).tolist()
    ref_name = "reference_summary"
else:
    refs = df["context"].fillna("").astype(str).tolist()  # weak reference if gold not available
    ref_name = "context_as_reference"

print("Using", ref_name, "as reference.")


Using context_as_reference as reference.


In [ ]:
rouge_results = {}
for col in CANDIDATE_SUMMARY_COLS:
    cands = df[col].fillna("").astype(str).tolist()
    rouge_results[col] = compute_rouge(cands, refs)

bertscore_results = {}
for col in CANDIDATE_SUMMARY_COLS:
    cands = df[col].fillna("").astype(str).tolist()
    P, R, F1 = bertscore(cands, refs, lang="en", rescale_with_baseline=False)
    bertscore_results[col] = {"P": float(P.mean()), "R": float(R.mean()), "F1": float(F1.mean())}

print("ROUGE:", pd.DataFrame(rouge_results))
print("BERTScore:", pd.DataFrame(bertscore_results))


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: ht

ROUGE:       extractive_summary  pegasus_summary  hybrid_blend
r1_p            1.000000         0.941414      0.996367
r1_r            0.320931         0.045502      0.113550
r1_f            0.448003         0.077339      0.166939
r2_p            0.992971         0.705064      0.990453
r2_r            0.318313         0.033668      0.112321
r2_f            0.444410         0.057398      0.165134
rl_p            0.994160         0.888265      0.992678
rl_r            0.319199         0.041128      0.112993
rl_f            0.445406         0.070380      0.166105
BERTScore:     extractive_summary  pegasus_summary  hybrid_blend
P             0.846536         0.888932      0.864412
R             0.837508         0.783917      0.815833
F1            0.841902         0.832906      0.839191
